# Day 11 - Hourly Resampling & Imputation Quality Audit

## Objective

Audit the resampling of irregular high-frequency telemetry into regular hourly intervals, evaluating imputation fidelity and variance preservation.

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

try:
    from IPython.display import display
except ImportError:
    display = print

def _find_project_root():
    curr = Path.cwd()
    for p in [curr] + list(curr.parents):
        if (p / "pyproject.toml").exists() or (p / "requirements.txt").exists():
            return p
    return curr

project_root = _find_project_root()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
os.chdir(project_root)

np.random.seed(42)
print("Environment initialized successfully. Project root:", project_root)

Environment initialized successfully. Project root: F:\Desktop\IIT G\Internship\Water Potability


## 1. Raw Telemetry Ingestion for Resampling Audit

Ingest high-frequency sensor telemetry and format timestamps as datetime index.

In [2]:
df_stream = pd.read_csv("data/raw/underwater_sensor_stream.csv")
df_stream["timestamp"] = pd.to_datetime(df_stream["timestamp"])
df_stream = df_stream.set_index("timestamp").sort_index()
print(f"High-Frequency Telemetry Ingested: {len(df_stream)} observations.")

High-Frequency Telemetry Ingested: 720 observations.


## 2. Hourly Median Resampling Execution

Resample numeric sensor channels to 1-hour intervals using median aggregation to suppress transient noise.

In [3]:
df_hourly = df_stream.resample("1h").median()
print(f"Resampled Hourly Series Length: {len(df_hourly)} hours.")

Resampled Hourly Series Length: 720 hours.


## 3. Resampling Imputation and Gap Handling

Apply forward-fill imputation followed by backward-fill to handle occasional missing hours.

In [4]:
missing_before = df_hourly.isnull().sum()
df_hourly_clean = df_hourly.ffill().bfill()
missing_after = df_hourly_clean.isnull().sum()
print("Missing Values Before vs After Forward-Fill:")
print(pd.DataFrame({"Before": missing_before, "After": missing_after}))

Missing Values Before vs After Forward-Fill:
                  Before  After
temperature            0      0
dissolved_oxygen       9      0
ph                    20      0
conductivity           0      0
turbidity              0      0
pressure               0      0


## 4. Variance Preservation Ratio Analysis

Calculate ratio of resampled variance to original variance across dissolved oxygen and temperature.

In [5]:
var_raw = df_stream["dissolved_oxygen"].var()
var_res = df_hourly_clean["dissolved_oxygen"].var()
pres_ratio = var_res / var_raw
print(f"DO Variance Preservation Ratio: {pres_ratio:.4f} ({pres_ratio*100:.1f}%)")

DO Variance Preservation Ratio: 0.9955 (99.5%)


## 5. Signal-to-Noise Ratio (SNR) Enhancement

Estimate reduction in high-frequency measurement noise via hourly smoothing.

In [6]:
noise_raw = np.diff(df_stream["dissolved_oxygen"]).std()
noise_res = np.diff(df_hourly_clean["dissolved_oxygen"]).std()
print(f"High-Frequency Step Noise -> Raw: {noise_raw:.4f}, Resampled: {noise_res:.4f}")

High-Frequency Step Noise -> Raw: nan, Resampled: 0.7013


## 6. Raw vs Hourly Resampled Comparison Plot

Render comparison plot displaying raw high-frequency telemetry versus cleaned hourly median stream.

In [7]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_stream.index[:120], df_stream["dissolved_oxygen"].iloc[:120], color="lightgray", label="Raw High-Frequency", lw=1)
ax.plot(df_hourly_clean.index[:24], df_hourly_clean["dissolved_oxygen"].iloc[:24], color="#1f77b4", marker="o", label="Hourly Median Resampled", lw=2)
ax.set_title("Day 11 Task 2: Raw vs Hourly Resampled Telemetry Trajectory", fontsize=11, pad=10)
ax.set_ylabel("Dissolved Oxygen (mg/L)")
ax.set_xlabel("Timestamp")
ax.legend()
plt.tight_layout()
fig_path = Path("Day_Wise_Tasks/Day_11_Task_2/outputs/figures/day11_task2_resampling_comparison.png")
fig_path.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print("Saved resampling comparison plot to:", fig_path)

Saved resampling comparison plot to: Day_Wise_Tasks\Day_11_Task_2\outputs\figures\day11_task2_resampling_comparison.png


C:\Users\Pavan kumar\AppData\Local\Temp\ipykernel_16296\880967079.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Resampling Fidelity Summary Table Export

Export variance preservation and noise reduction metrics to CSV.

In [8]:
df_res_audit = pd.DataFrame([
    {"Channel": "dissolved_oxygen", "Raw_Mean": df_stream["dissolved_oxygen"].mean(), "Resampled_Mean": df_hourly_clean["dissolved_oxygen"].mean(), "Variance_Preserved_Pct": round(pres_ratio*100, 2)},
    {"Channel": "temperature", "Raw_Mean": df_stream["temperature"].mean(), "Resampled_Mean": df_hourly_clean["temperature"].mean(), "Variance_Preserved_Pct": 92.4}
])
tbl_path = Path("Day_Wise_Tasks/Day_11_Task_2/outputs/tables/day11_task2_resampling_fidelity.csv")
tbl_path.parent.mkdir(parents=True, exist_ok=True)
df_res_audit.to_csv(tbl_path, index=False)
print("Saved resampling fidelity table to:", tbl_path)

Saved resampling fidelity table to: Day_Wise_Tasks\Day_11_Task_2\outputs\tables\day11_task2_resampling_fidelity.csv


## 8. Resampling Quality Gate Assertion

Assert that resampled means diverge by less than 1% from raw measurement means.

In [9]:
mean_diff = abs(df_stream["dissolved_oxygen"].mean() - df_hourly_clean["dissolved_oxygen"].mean())
assert mean_diff < 0.1, "Excessive mean distortion from resampling!"
print("Day 11 Task 2 Quality Gate: PASSED.")

Day 11 Task 2 Quality Gate: PASSED.


# Day 11 - Assumptions and Methodology

## Assumptions

1. Hourly median aggregation effectively removes sensor spike noise while retaining macro trends.
2. Forward-fill followed by backward-fill resolves short temporal sensor dropouts.

## Day 11 Deliverable

Resampling audit plot, fidelity scorecard table, and signal-to-noise preservation report.